# Genned: adversarial fine-tuning (Phase 2b, issue #15)

Fine-tunes the app's two detectors so that invisible perturbations up to **4/255** can't flip them
(PGD adversarial training, half clean / half attacked batches). Runs on a free **Colab or Kaggle GPU**.

**Before you start**
1. *Runtime → Change runtime type → GPU* (Colab), or *Accelerator → GPU T4/P100* (Kaggle; also turn *Internet* on).
2. Add a secret named **`HF_TOKEN`**: a Hugging Face token with **write** access, from an account that has accepted
   the gated `Dafilab/ai-image-detector` and `DF26` terms (the same token the repo's CI uses works if it has write scope).
   Colab: the key icon in the left sidebar. Kaggle: *Add-ons → Secrets*.
3. *Run all.* About **1–1.5 h** on a T4: data download ~10 min, Community Forensics ~15 min, EfficientNet-B4 ~45–75 min.

**What leaves this machine:** only the fine-tuned weights and a JSON of numbers, uploaded to a **private** Hugging Face
repo `<your-username>/genned-robust`. The training images are public datasets and stay on this VM.
Training uses the datasets' **train** splits only; every benchmark in the repo uses the **test** splits.

When it finishes, tell Claude the repo name. CI then builds, calibrates and benchmarks the robust ensemble;
nothing ships unless it passes the gates in `internal-docs/MODEL.md`.

In [ ]:
!nvidia-smi || echo 'No GPU: switch the runtime to a GPU before continuing.'

In [ ]:
import os, subprocess

def secret(name):
    try:
        from google.colab import userdata  # Colab
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add an HF_TOKEN secret (write scope) first - see the first cell."
gh = secret("GITHUB_TOKEN")  # only needed if the repo is private
url = f"https://{gh}@github.com/as791/genned.git" if gh else "https://github.com/as791/genned.git"
if not os.path.isdir("genned"):
    subprocess.run(["git", "clone", "--depth", "1", url, "genned"], check=True)
os.chdir("genned")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "timm>=1.0,<2" "huggingface_hub>=0.23,<1" "datasets>=2.19,<5" "onnxruntime>=1.18,<2" "pillow>=10,<12"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/genned-robust"
print("Weights will be uploaded (privately) to", HF_REPO)

## Training data (train splits only) and validation (Defactify validation split)
`--strict-split` makes the download fail rather than silently fall back to the test split the benchmarks use.

In [ ]:
PER_CLASS = 2000  # images per class per dataset; lower it if the session is short on time or disk

!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split train --strict-split --out data/train/defactify --per-class {PER_CLASS} --max-scan 15000 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6
!python tools/fetch_eval_data.py --dataset julienlucas/midjourney-dalle-sd-nanobananapro-dataset \
    --split train --strict-split --out data/train/mj-dalle-sd-nbp --per-class {PER_CLASS} --max-scan 15000
!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split validation --strict-split --out data/val/defactify --per-class 150 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6

## 1. Community Forensics ViT-S 224 (~15 min on a T4)

In [ ]:
!python tools/adv_finetune.py --model commfor \
    --train-data data/train/defactify data/train/mj-dalle-sd-nbp --val-data data/val/defactify \
    --per-class {PER_CLASS} --epochs 2 --eps 4 --out runs --push-to-hub {HF_REPO}

## 2. Dafilab EfficientNet-B4 (~45–75 min on a T4)
If the session is about to time out, you can stop after step 1: CI can build an ensemble with only the
Community Forensics model fine-tuned.

In [ ]:
!python tools/adv_finetune.py --model bundled \
    --train-data data/train/defactify data/train/mj-dalle-sd-nbp --val-data data/val/defactify \
    --per-class {PER_CLASS} --epochs 2 --eps 4 --out runs --push-to-hub {HF_REPO}

In [ ]:
import json, glob
for path in sorted(glob.glob("runs/*-train-log.json")):
    log = json.load(open(path))
    print(f"{log['model']}: selected epoch {log['selected_epoch']}, clean gate met: {log['meets_clean_gate']}")
    for h in log["history"]:
        print(f"  epoch {h['epoch']}: clean AUC {h['clean_auc']:.3f}, robust acc @4/255 {h['robust_acc']:.1%}")
print()
print("Done. Tell Claude:", HF_REPO)